# AADS-ULoRA Colab: Performance Monitoring

This notebook provides comprehensive performance monitoring and analysis for the training pipeline.

## What this notebook does:
1. **Load training logs** - Parse CSV and JSON logs
2. **Analyze metrics** - Loss, accuracy, memory usage
3. **Compare experiments** - Multiple training runs
4. **Generate reports** - Performance summaries and visualizations
5. **Optimization insights** - Recommendations for improvement

---
**Expected Runtime:** 10-20 minutes
**Required:** Completed training logs

---

## 1. Setup and Configuration

In [ ]:
import sys
from pathlib import Path
import json
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import glob

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Add workspace to path
workspace_dir = Path('/content/aads_ulora')
sys.path.insert(0, str(workspace_dir))

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Setup complete")
print(f"Workspace: {workspace_dir}")

## 2. Load Training Logs

In [ ]:
# Logs directory
logs_dir = workspace_dir / 'logs'
checkpoints_dir = workspace_dir / 'checkpoints'
outputs_dir = workspace_dir / 'outputs'

print("Looking for training logs...")

# Find all history files
history_files = {
    'phase1': logs_dir / 'phase1_history.json',
    'phase2': logs_dir / 'phase2_history.json',
    'phase3': logs_dir / 'phase3_history.json'
}

# Load histories
histories = {}
for phase, path in history_files.items():
    if path.exists():
        with open(path, 'r') as f:
            histories[phase] = json.load(f)
        print(f"✅ Loaded {phase} history: {len(histories[phase].get('train_loss', []))} epochs")
    else:
        print(f"❌ {phase} history not found: {path}")

# Load OOD metrics if available
ood_metrics_path = logs_dir / 'ood_metrics.json'
if ood_metrics_path.exists():
    with open(ood_metrics_path, 'r') as f:
        ood_metrics = json.load(f)
    print(f"✅ Loaded OOD metrics")
else:
    ood_metrics = None
    print(f"⚠️  OOD metrics not found")

# Load test summary if available
test_summary_path = outputs_dir / 'test_summary.json'
if test_summary_path.exists():
    with open(test_summary_path, 'r') as f:
        test_summary = json.load(f)
    print(f"✅ Loaded test summary")
else:
    test_summary = None
    print(f"⚠️  Test summary not found")

## 3. Training Curves Analysis

In [ ]:
def plot_training_curves(histories):
    """Plot comprehensive training curves."""
    phases = list(histories.keys())
    n_phases = len(phases)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    colors = {'phase1': '#3498db', 'phase2': '#2ecc71', 'phase3': '#e74c3c'}
    
    # Loss curves
    ax_idx = 0
    for phase in phases:
        history = histories[phase]
        epochs = range(1, len(history['train_loss']) + 1)
        axes[ax_idx].plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
        axes[ax_idx].plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
        axes[ax_idx].set_xlabel('Epoch')
        axes[ax_idx].set_ylabel('Loss')
        axes[ax_idx].set_title(f'{phase.capitalize()} - Loss Curves')
        axes[ax_idx].legend()
        axes[ax_idx].grid(True, alpha=0.3)
        ax_idx += 1
    
    # Accuracy curves (if available)
    for i, phase in enumerate(phases):
        history = histories[phase]
        if 'train_accuracy' in history and 'val_accuracy' in history:
            epochs = range(1, len(history['train_accuracy']) + 1)
            axes[ax_idx].plot(epochs, history['train_accuracy'], 'b-', label='Train Acc', linewidth=2)
            axes[ax_idx].plot(epochs, history['val_accuracy'], 'r-', label='Val Acc', linewidth=2)
            axes[ax_idx].set_xlabel('Epoch')
            axes[ax_idx].set_ylabel('Accuracy')
            axes[ax_idx].set_title(f'{phase.capitalize()} - Accuracy')
            axes[ax_idx].legend()
            axes[ax_idx].grid(True, alpha=0.3)
        ax_idx += 1
    
    # GPU Memory usage (if available)
    for i, phase in enumerate(phases):
        history = histories[phase]
        if 'gpu_memory' in history and history['gpu_memory']:
            memory_values = [m['allocated_gb'] for m in history['gpu_memory']]
            axes[ax_idx].plot(memory_values, color=colors[phase], linewidth=2)
            axes[ax_idx].set_xlabel('Step')
            axes[ax_idx].set_ylabel('GPU Memory (GB)')
            axes[ax_idx].set_title(f'{phase.capitalize()} - GPU Memory')
            axes[ax_idx].grid(True, alpha=0.3)
        ax_idx += 1
    
    # Hide unused subplots
    for idx in range(ax_idx, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

# Plot training curves
if histories:
    plot_training_curves(histories)
    print("✅ Training curves plotted")
else:
    print("❌ No training histories available")

## 4. Phase Comparison

In [ ]:
# Compare final metrics across phases
if histories:
    comparison_data = []
    
    for phase, history in histories.items():
        if history.get('train_loss') and history.get('val_loss'):
            final_train_loss = history['train_loss'][-1]
            final_val_loss = history['val_loss'][-1]
            best_val_loss = min(history['val_loss'])
            best_val_epoch = history['val_loss'].index(best_val_loss) + 1
            
            # Get accuracy if available
            final_acc = history.get('val_accuracy', [None])[-1]
            best_acc = max(history.get('val_accuracy', [0])) if history.get('val_accuracy') else None
            
            comparison_data.append({
                'Phase': phase.capitalize(),
                'Final Train Loss': final_train_loss,
                'Final Val Loss': final_val_loss,
                'Best Val Loss': best_val_loss,
                'Best Val Epoch': best_val_epoch,
                'Final Accuracy': final_acc,
                'Best Accuracy': best_acc,
                'Epochs': len(history['train_loss'])
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\n" + "="*80)
    print("📊 PHASE COMPARISON")
    print("="*80)
    print(comparison_df.to_string(index=False))
    
    # Save comparison
    comparison_path = outputs_dir / 'phase_comparison.csv'
    comparison_df.to_csv(comparison_path, index=False)
    print(f"\n✅ Comparison saved to: {comparison_path}")
    
    # Plot comparison
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Loss comparison
    x = np.arange(len(comparison_df))
    width = 0.35
    axes[0].bar(x - width/2, comparison_df['Final Train Loss'], width, label='Train Loss', alpha=0.8)
    axes[0].bar(x + width/2, comparison_df['Final Val Loss'], width, label='Val Loss', alpha=0.8)
    axes[0].set_xlabel('Phase')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Final Loss by Phase')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(comparison_df['Phase'])
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Best loss comparison
    axes[1].bar(comparison_df['Phase'], comparison_df['Best Val Loss'], color='#2ecc71', alpha=0.8)
    axes[1].set_xlabel('Phase')
    axes[1].set_ylabel('Best Validation Loss')
    axes[1].set_title('Best Validation Loss')
    axes[1].grid(True, alpha=0.3)
    
    # Accuracy comparison (if available)
    if 'Final Accuracy' in comparison_df.columns and comparison_df['Final Accuracy'].notna().any():
        axes[2].bar(comparison_df['Phase'], comparison_df['Final Accuracy'], color='#3498db', alpha=0.8)
        axes[2].set_xlabel('Phase')
        axes[2].set_ylabel('Accuracy')
        axes[2].set_title('Final Accuracy')
        axes[2].set_ylim(0, 1)
        axes[2].grid(True, alpha=0.3)
    else:
        axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("❌ No histories available for comparison")

## 5. Memory Usage Analysis

In [ ]:
# Analyze GPU memory usage across phases
if histories:
    memory_data = []
    
    for phase, history in histories.items():
        if 'gpu_memory' in history and history['gpu_memory']:
            memory_values = [m['allocated_gb'] for m in history['gpu_memory']]
            memory_data.append({
                'Phase': phase.capitalize(),
                'Mean Memory (GB)': np.mean(memory_values),
                'Max Memory (GB)': np.max(memory_values),
                'Min Memory (GB)': np.min(memory_values),
                'Std Memory (GB)': np.std(memory_values),
                'Measurements': len(memory_values)
            })
    
    if memory_data:
        memory_df = pd.DataFrame(memory_data)
        print("\n" + "="*80)
        print("💾 MEMORY USAGE ANALYSIS")
        print("="*80)
        print(memory_df.to_string(index=False))
        
        # Plot memory usage
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # Bar chart of mean memory
        axes[0].bar(memory_df['Phase'], memory_df['Mean Memory (GB)'], alpha=0.8)
        axes[0].set_xlabel('Phase')
        axes[0].set_ylabel('Mean GPU Memory (GB)')
        axes[0].set_title('Average Memory Usage by Phase')
        axes[0].grid(True, alpha=0.3)
        
        # Box plot comparison
        all_memory = []
        labels = []
        for phase, history in histories.items():
            if 'gpu_memory' in history and history['gpu_memory']:
                memory_values = [m['allocated_gb'] for m in history['gpu_memory']]
                all_memory.append(memory_values)
                labels.append(phase.capitalize())
        
        axes[1].boxplot(all_memory, labels=labels)
        axes[1].set_xlabel('Phase')
        axes[1].set_ylabel('GPU Memory (GB)')
        axes[1].set_title('Memory Distribution')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  No memory usage data available")
else:
    print("❌ No histories available for memory analysis")

## 6. OOD Detection Analysis

In [ ]:
# Analyze OOD detection performance
if ood_metrics:
    print("\n" + "="*80)
    print("🔍 OOD DETECTION ANALYSIS")
    print("="*80)
    
    # Extract OOD metrics
    if 'ood_metrics_history' in ood_metrics and ood_metrics['ood_metrics_history']:
        ood_history = ood_metrics['ood_metrics_history']
        
        # Compute average metrics
        avg_proto_dist = np.mean([m['prototype_distances'].mean() for m in ood_history if 'prototype_distances' in m])
        avg_maha_score = np.mean([m.get('mahalanobis_scores', torch.tensor(0)).mean() for m in ood_history])
        avg_proto_anomaly = np.mean([m['prototype_anomaly'].mean() for m in ood_history])
        avg_maha_anomaly = np.mean([m.get('mahalanobis_anomaly', torch.zeros(1)).mean() for m in ood_history])
        
        print(f"\nAverage OOD Metrics:")
        print(f"  Prototype distance: {avg_proto_dist:.4f}")
        print(f"  Mahalanobis score: {avg_maha_score:.4f}")
        print(f"  Prototype OOD rate: {avg_proto_anomaly:.2%}")
        print(f"  Mahalanobis OOD rate: {avg_maha_anomaly:.2%}")
    
    # Prototype analysis
    if 'prototype_embeddings' in ood_metrics and ood_metrics['prototype_embeddings']:
        prototypes = np.array(ood_metrics['prototype_embeddings'])
        print(f"\nPrototype Analysis:")
        print(f"  Number of prototypes: {prototypes.shape[0]}")
        print(f"  Prototype dimension: {prototypes.shape[1]}")
        
        # Compute pairwise distances between prototypes
        from scipy.spatial.distance import pdist, squareform
        proto_distances = pdist(prototypes, metric='euclidean')
        print(f"  Mean inter-prototype distance: {np.mean(proto_distances):.4f}")
        print(f"  Min inter-prototype distance: {np.min(proto_distances):.4f}")
        print(f"  Max inter-prototype distance: {np.max(proto_distances):.4f}")
        
        # Plot prototype distances
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # Histogram of pairwise distances
        axes[0].hist(proto_distances, bins=20, alpha=0.7, edgecolor='black')
        axes[0].set_xlabel('Euclidean Distance')
        axes[0].set_ylabel('Count')
        axes[0].set_title('Inter-Prototype Distances')
        axes[0].axvline(np.mean(proto_distances), color='red', linestyle='--', label=f'Mean: {np.mean(proto_distances):.3f}')
        axes[0].legend()
        
        # Heatmap of pairwise distances
        dist_matrix = squareform(proto_distances)
        im = axes[1].imshow(dist_matrix, cmap='viridis', aspect='auto')
        axes[1].set_xlabel('Prototype Index')
        axes[1].set_ylabel('Prototype Index')
        axes[1].set_title('Prototype Distance Matrix')
        plt.colorbar(im, ax=axes[1])
        
        plt.tight_layout()
        plt.show()
else:
    print("⚠️  No OOD metrics available for analysis")

## 7. Test Results Analysis

In [ ]:
# Load and analyze test predictions
predictions_path = outputs_dir / 'predictions.csv'

if predictions_path.exists():
    predictions_df = pd.read_csv(predictions_path)
    
    print("\n" + "="*80)
    print("📈 TEST RESULTS ANALYSIS")
    print("="*80)
    
    # Overall metrics
    total_samples = len(predictions_df)
    correct_predictions = predictions_df['correct'].sum()
    accuracy = correct_predictions / total_samples
    avg_confidence = predictions_df['confidence'].mean()
    
    print(f"\nOverall Performance:")
    print(f"  Total samples: {total_samples}")
    print(f"  Correct predictions: {correct_predictions}")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Average confidence: {avg_confidence:.4f}")
    
    # Per-class metrics
    print(f"\nPer-Class Performance:")
    class_metrics = predictions_df.groupby('true_class').agg({
        'correct': ['sum', 'count'],
        'confidence': 'mean'
    }).round(4)
    class_metrics.columns = ['correct', 'total', 'avg_confidence']
    class_metrics['accuracy'] = class_metrics['correct'] / class_metrics['total']
    print(class_metrics.to_string())
    
    # Plot performance by class
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Accuracy by class
    class_metrics_sorted = class_metrics.sort_values('accuracy', ascending=False)
    axes[0].barh(range(len(class_metrics_sorted)), class_metrics_sorted['accuracy'])
    axes[0].set_yticks(range(len(class_metrics_sorted)))
    axes[0].set_yticklabels(class_metrics_sorted.index)
    axes[0].set_xlabel('Accuracy')
    axes[0].set_title('Accuracy by Class')
    axes[0].set_xlim(0, 1)
    axes[0].grid(True, alpha=0.3, axis='x')
    
    # Confidence distribution
    axes[1].hist(predictions_df['confidence'], bins=20, alpha=0.7, edgecolor='black')
    axes[1].axvline(avg_confidence, color='red', linestyle='--', label=f'Mean: {avg_confidence:.3f}')
    axes[1].set_xlabel('Confidence')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Confidence Distribution')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Confidence vs correctness
    correct_conf = predictions_df[predictions_df['correct']]['confidence']
    incorrect_conf = predictions_df[~predictions_df['correct']]['confidence']
    axes[2].boxplot([correct_conf, incorrect_conf], labels=['Correct', 'Incorrect'])
    axes[2].set_ylabel('Confidence')
    axes[2].set_title('Confidence by Correctness')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Save class metrics
    class_metrics_path = outputs_dir / 'class_metrics.csv'
    class_metrics.to_csv(class_metrics_path)
    print(f"\n✅ Class metrics saved to: {class_metrics_path}")
else:
    print("⚠️  Predictions file not found. Run the testing notebook first.")

## 8. Optimization Insights

In [ ]:
# Generate optimization recommendations
print("\n" + "="*80)
print("💡 OPTIMIZATION INSIGHTS")
print("="*80)

insights = []

# Analyze training convergence
if histories:
    for phase, history in histories.items():
        if 'train_loss' in history and 'val_loss' in history:
            train_loss = history['train_loss']
            val_loss = history['val_loss']
            
            # Check for overfitting
            if train_loss[-1] < val_loss[-1] * 0.8:
                insights.append(f"{phase.capitalize()}: Possible overfitting (train loss much lower than val loss)")
            
            # Check for underfitting
            if len(train_loss) > 5 and abs(train_loss[-1] - train_loss[-5]) < 0.001:
                insights.append(f"{phase.capitalize()}: Training may be stuck (loss not decreasing)")
            
            # Check for rapid convergence
            if len(train_loss) >= 3 and train_loss[-1] / train_loss[0] < 0.1:
                insights.append(f"{phase.capitalize()}: Rapid convergence detected (consider early stopping)")

# Analyze memory usage
if memory_data:
    for mem in memory_data:
        if mem['Max Memory (GB)'] > 12:  # Assuming Colab has ~12-15GB
            insights.append(f"{mem['Phase']}: High memory usage ({mem['Max Memory (GB)']:.2f}GB). Consider reducing batch size.")

# Analyze accuracy
if test_summary and 'results' in test_summary:
    results = test_summary['results']
    for phase in ['phase1_accuracy', 'phase2_accuracy', 'phase3_accuracy']:
        if phase in results and results[phase] < 0.7:
            insights.append(f"{phase.split('_')[0].capitalize()}: Low accuracy ({results[phase]:.2%}). Consider more training or data augmentation.")

# Print insights
if insights:
    for i, insight in enumerate(insights, 1):
        print(f"{i}. {insight}")
else:
    print("No significant issues detected. Training appears healthy!")

# Generate summary report
print("\n" + "="*80)
print("📋 PERFORMANCE SUMMARY")
print("="*80)

summary = {
    'timestamp': datetime.now().isoformat(),
    'phases_trained': list(histories.keys()),
    'total_epochs': {phase: len(history.get('train_loss', [])) for phase, history in histories.items()},
    'final_metrics': {},
    'insights': insights
}

for phase, history in histories.items():
    if history.get('train_loss') and history.get('val_loss'):
        summary['final_metrics'][phase] = {
            'final_train_loss': history['train_loss'][-1],
            'final_val_loss': history['val_loss'][-1],
            'best_val_loss': min(history['val_loss']),
            'best_val_epoch': history['val_loss'].index(min(history['val_loss'])) + 1
        }
        if 'val_accuracy' in history:
            summary['final_metrics'][phase]['final_accuracy'] = history['val_accuracy'][-1]
            summary['final_metrics'][phase]['best_accuracy'] = max(history['val_accuracy'])

# Add test results if available
if test_summary and 'results' in test_summary:
    summary['test_results'] = test_summary['results']

# Save summary
summary_path = outputs_dir / 'performance_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\n✅ Summary saved to: {summary_path}")

print("\n" + "="*80)
print("✅ Performance Monitoring Complete!")
print("="*80)